# EDA Online Retail Dataset
## Exploratory Data Analysis - Data Cleaning & Preparation

**Dataset:** Online Retail Transaction Data  
**Sumber:** UCI Machine Learning Repository  
**Tanggal Analisis:** 2026-03-28  
**Tujuan:** Membersihkan dan menyiapkan data untuk analisis di Power BI

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [ ]:
# Load dataset
df = pd.read_excel('../data/raw/Online Retail.xlsx')

print('='*60)
print('DATA LOADING SUMMARY')
print('='*60)
print(f'Total Rows: {df.shape[0]:,}')
print(f'Total Columns: {df.shape[1]}')
print(f'Columns: {list(df.columns)}')

## 3. Initial Data Overview

In [ ]:
print('='*60)
print('INITIAL DATA OVERVIEW')
print('='*60)

print('\n--- First 5 Rows ---')
print(df.head())

print('\n--- Data Info ---')
print(df.info())

print('\n--- Data Types ---')
print(df.dtypes)

print('\n--- Descriptive Statistics ---')
print(df.describe())

## 4. Missing Values Analysis

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage (%)': missing_pct
})

print('='*60)
print('MISSING VALUES ANALYSIS')
print('='*60)
print(missing_df[missing_df['Missing Count'] > 0])

## 5. Data Cleaning

In [ ]:
print('='*60)
print('DATA CLEANING PROCESS')
print('='*60)

# 1. Handle missing values
print('\n[Step 1] Handling Missing Values...')
df['Description'] = df['Description'].fillna('Unknown')
df['CustomerID'] = df['CustomerID'].astype(str).fillna('Guest')
print(f"  - Description: {df['Description'].isnull().sum()} missing")
print(f"  - CustomerID: {df['CustomerID'].isnull().sum()} missing")

# 2. Remove duplicates (exact matches)
print('\n[Step 2] Removing Exact Duplicates...')
exact_dups = df.duplicated().sum()
df = df.drop_duplicates()
print(f"  - Removed {exact_dups} exact duplicate rows")

# 3. Aggregate data (groupby InvoiceNo + StockCode)
print('\n[Step 3] Aggregating Data (InvoiceNo + StockCode)...')
original_rows = df.shape[0]
df = df.groupby(['InvoiceNo', 'StockCode'], as_index=False).agg({
    'Description': 'first',
    'Quantity': 'sum',
    'InvoiceDate': 'first',
    'UnitPrice': 'first',
    'CustomerID': 'first',
    'Country': 'first'
})
print(f"  - Rows before aggregation: {original_rows:,}")
print(f"  - Rows after aggregation: {df.shape[0]:,}")
print(f"  - Rows reduced: {original_rows - df.shape[0]:,}")

# 4. Create Revenue column
print('\n[Step 4] Creating Revenue Column...')
df['Revenue'] = df['Quantity'] * df['UnitPrice']
print(f"  - Revenue column created: Quantity x UnitPrice")

# 5. Extract date features
print('\n[Step 5] Extracting Date Features...')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['MonthName'] = df['InvoiceDate'].dt.month_name()
print(f"  - Year, Month, Day, Hour, DayOfWeek, MonthName created")

print('\n' + '='*60)
print('CLEANING COMPLETE!')
print('='*60)

## 6. Descriptive Statistics (After Cleaning)

In [ ]:
print('='*60)
print('DESCRIPTIVE STATISTICS')
print('='*60)

print('\n--- Numeric Columns ---')
print(df.describe())

print('\n--- Categorical Columns ---')
print(df.describe(include=['object']))

## 7. Key Insights Summary

In [ ]:
print('='*60)
print('KEY INSIGHTS SUMMARY')
print('='*60)

print('\nGENERAL STATISTICS')
print(f"  - Total Transactions: {df['InvoiceNo'].nunique():,}")
print(f"  - Total Products: {df['StockCode'].nunique():,}")
print(f"  - Total Customers: {df[df['CustomerID'] != 'Guest']['CustomerID'].nunique():,}")
print(f"  - Guest Customers: {(df['CustomerID'] == 'Guest').sum():,}")
print(f"  - Total Revenue: GBP {df['Revenue'].sum():,.2f}")
print(f"  - Average Transaction Value: GBP {df.groupby('InvoiceNo')['Revenue'].sum().mean():.2f}")

print('\nTOP COUNTRIES')
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(5)
for i, (country, revenue) in enumerate(top_countries.items(), 1):
    print(f"  {i}. {country}: GBP {revenue:,.2f}")

print('\nTOP PRODUCTS (by Revenue)')
top_products = df[df['Revenue'] > 0].groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(5)
for i, (product, revenue) in enumerate(top_products.items(), 1):
    print(f"  {i}. {product[:50]}: GBP {revenue:,.2f}")

print('\nPEAK SALES')
peak_month = df.groupby('MonthName')['Revenue'].sum().idxmax()
peak_day = df.groupby('DayOfWeek')['Revenue'].sum().idxmax()
peak_hour = df.groupby('Hour')['Revenue'].sum().idxmax()
print(f"  - Peak Month: {peak_month}")
print(f"  - Peak Day: {peak_day}")
print(f"  - Peak Hour: {peak_hour}:00")

print('\n' + '='*60)

## 8. Save Cleaned Data

In [ ]:
# Select columns for cleaned dataset
columns_to_keep = [
    'InvoiceNo', 'StockCode', 'Description', 'Quantity', 
    'InvoiceDate', 'UnitPrice', 'Revenue', 'CustomerID', 
    'Country', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'MonthName'
]

df_clean = df[columns_to_keep].copy()

# Save to processed folder
output_path = '../data/processed/online_retail_clean.csv'
df_clean.to_csv(output_path, index=False)

print('='*60)
print('DATA SAVED SUCCESSFULLY!')
print('='*60)
print(f'\nOutput Path: {output_path}')
print(f'Total Rows: {df_clean.shape[0]:,}')
print(f'Total Columns: {df_clean.shape[1]}')
print(f'\nColumns: {list(df_clean.columns)}')
print('\nFirst 5 Rows:')
print(df_clean.head())
print('\nData types:')
print(df_clean.dtypes)

---
**End of Analysis**  
Data siap digunakan untuk visualisasi di Power BI.